In [3]:
import duckdb
import pandas as pd
import os

# Pfade definieren
data_path = r"C:\Users\Trainee\Desktop\DPP_Projekte\DPP-2026_Julian\data\raw\transfermarkt_data_2026-04-07"
players_file = os.path.join(data_path, "players.csv")
transfers_file = os.path.join(data_path, "transfers.csv")
clubs_file = os.path.join(data_path, "clubs.csv")

# Die korrigierte SQL-Abfrage mit Joins auf die Clubs-Tabelle
query = f"""
SELECT 
    p.player_id,
    p.name,
    p.position,
    p.sub_position,
    (date_part('year', t.transfer_date) - date_part('year', p.date_of_birth)) as age_at_transfer,
    t.transfer_fee as einkaufspreis,
    p.highest_market_value_in_eur as peak_marktwert,
    c_from.domestic_competition_id as origin_league_id,
    t.transfer_date
FROM read_csv_auto('{transfers_file}') t
JOIN read_csv_auto('{players_file}') p ON t.player_id = p.player_id
JOIN read_csv_auto('{clubs_file}') c_to ON t.to_club_id = c_to.club_id
JOIN read_csv_auto('{clubs_file}') c_from ON t.from_club_id = c_from.club_id
WHERE c_to.domestic_competition_id = 'L1'      -- Ziel: Bundesliga
AND c_from.domestic_competition_id != 'L1'     -- Herkunft: Ausland
AND t.transfer_fee >= 100000                   -- Nur Profi-relevante Summen
AND t.transfer_date > '2015-01-01'
"""

df_raw = duckdb.query(query).to_df()

# Dubletten entfernen
df_clean = df_raw.sort_values('transfer_date').drop_duplicates(subset=['player_id'], keep='first').copy()

# ROI Metriken
df_clean['absolute_growth'] = df_clean['peak_marktwert'] - df_clean['einkaufspreis']
df_clean['roi_pct'] = (df_clean['absolute_growth'] / df_clean['einkaufspreis']) * 100
df_clean['roi_score'] = df_clean['roi_pct'] / df_clean['age_at_transfer']

print(f"Datensatz bereit: {len(df_clean)} Transfers identifiziert.")
display(df_clean.head())

Datensatz bereit: 535 Transfers identifiziert.


,player_id,name,position,sub_position,age_at_transfer,einkaufspreis,peak_marktwert,origin_league_id,transfer_date,absolute_growth,roi_pct,roi_score
269,111078,Emil Forsberg,Attack,Left Winger,24,3700000.0,28000000,SE1,2015-01-07,24300000.0,656.756757,27.364865
350,206386,Tin Jedvaj,Defender,Centre-Back,20,7000000.0,9000000,IT1,2015-01-20,2000000.0,28.571429,1.428571
371,230541,Yoshinori Muto,Attack,Centre-Forward,23,2800000.0,10000000,JAP1,2015-07-01,7200000.0,257.142857,11.180124
232,33210,Przemyslaw Tyton,Goalkeeper,Goalkeeper,28,1000000.0,5000000,NL1,2015-07-01,4000000.0,400.000000,14.285714
239,55508,Lewis Holtby,Midfield,Central Midfield,25,6500000.0,11000000,GB1,2015-07-01,4500000.0,69.230769,2.769231


In [ ]:
# Analyse der Marktwert-Rendite (ROI) von ausländischen Neuzugängen der Bundesliga seit dem 01.01.2015 (Ablöse ≥ 100k €), gewichtet nach dem Transferalter (ROI-Score).

# Hinweis zum ROI-Score: 
# prozentuale Marktwertsteigerung in Prozent / Alter bei Wechsel. Die Analyse misst die maximale Wertschöpfung. Der Zeitraum ist definiert als die Dauer vom Transfer bis zum Erreichen des Peak-Marktwerts. 
# Durch die Division durch das Transfer-Alter wird der Faktor Zeit indirekt berücksichtigt, da jüngere Talente ein längeres Entwicklungsfenster bis zum Peak aufweisen.

# Hinweis zur Methodik: 
# Wir nutzen den 'peak_marktwert' der gesamten Karriere als Proxy für das Scouting-Potenzial einer Region. 
# Dies misst die maximale Qualitätsentwicklung eines Talents nach dem Wechsel nach Deutschland, unabhängig davon, ob dieser Peak noch während der Bundesliga-Zeit oder erst beim nächsten Verein erreicht wurde.

# Next Step: Regionen gruppieren, um Übersicht zu verbessern.

In [4]:
# Aggregation auf Liga-Ebene (ungruppiert)
league_analysis = df_clean.groupby('origin_league_id').agg({
    'player_id': 'count',
    'einkaufspreis': 'mean',
    'peak_marktwert': 'mean',
    'roi_pct': 'mean',
    'roi_score': 'mean'
}).rename(columns={'player_id': 'Anzahl_Transfers'}).sort_values('roi_score', ascending=False)

# Formatierung für bessere Lesbarkeit
league_analysis['Ø_Einkaufspreis'] = league_analysis['einkaufspreis'].map('{:,.0f} €'.format)
league_analysis['Ø_Peak_Marktwert'] = league_analysis['peak_marktwert'].map('{:,.0f} €'.format)
league_analysis['Ø_ROI_%'] = league_analysis['roi_pct'].round(1).astype(str) + '%'
league_analysis['ROI_Score'] = league_analysis['roi_score'].round(2)

# Nur die relevanten Spalten anzeigen
display(league_analysis[['Anzahl_Transfers', 'Ø_Einkaufspreis', 'Ø_Peak_Marktwert', 'Ø_ROI_%', 'ROI_Score']])

,Anzahl_Transfers,Ø_Einkaufspreis,Ø_Peak_Marktwert,Ø_ROI_%,ROI_Score
origin_league_id,,,,,
NO1,7,"1,321,429 €","5,771,429 €",2447.1%,116.36
BE1,40,"6,393,750 €","13,562,500 €",393.5%,17.89
RSK1,4,"525,000 €","2,050,000 €",508.4%,17.74
PO1,15,"10,734,667 €","21,233,333 €",457.1%,16.06
RU1,3,"3,666,667 €","15,833,333 €",405.6%,15.34
JAP1,9,"1,394,444 €","6,766,667 €",362.7%,15.13
BRA1,8,"8,487,500 €","17,062,500 €",278.0%,14.27
FR1,84,"10,643,452 €","24,669,048 €",297.9%,13.71
ES1,24,"11,229,167 €","22,916,667 €",287.9%,12.92


In [ ]:
# Erkenntnisse: 
# Norwegen fällt auf als extremer Ausreißer (ROI-Score sechs mal so groß wie Zweitplatzierter Belgien).
# Ligen mit weniger als 10 Transfers werden genauer analysiert.
# Ligen mit extrem wenigen Transfers (n < 6) haben kaum Aussagekraft und werden im nächsten Schritt rausgefiltert.

In [5]:
# 1. Schritt: Ligen mit weniger als 6 Transfers identifizieren und entfernen
league_counts = df_clean['origin_league_id'].value_counts()
leagues_to_keep = league_counts[league_counts >= 6].index

df_filtered = df_clean[df_clean['origin_league_id'].isin(leagues_to_keep)].copy()

# 2. Schritt: Ligen mit 6 bis 9 Transfers für die Ausreißer-Analyse isolieren
outlier_check_leagues = league_counts[(league_counts >= 6) & (league_counts < 10)].index
df_outlier_check = df_filtered[df_filtered['origin_league_id'].isin(outlier_check_leagues)].copy()

# 3. Den Outlier-Check DataFrame zur Ansicht vorbereiten
# Wir sortieren nach Liga und ROI-Score, um die "Spitzen" sofort zu sehen
df_outlier_check_display = df_outlier_check[[
    'origin_league_id', 'name', 'age_at_transfer', 
    'einkaufspreis', 'peak_marktwert', 'roi_score'
]].sort_values(['origin_league_id', 'roi_score'], ascending=[True, False])

print(f"Ligen nach Filterung (n>=6): {len(leagues_to_keep)}")
print(f"Gesamtanzahl Transfers im gefilterten Set: {len(df_filtered)}")
print("\n--- EINZELANSICHT: Ligen mit 6 bis 9 Transfers (Potenzielle Ausreißer) ---")
display(df_outlier_check_display)

Ligen nach Filterung (n>=6): 22
Gesamtanzahl Transfers im gefilterten Set: 521

--- EINZELANSICHT: Ligen mit 6 bis 9 Transfers (Potenzielle Ausreißer) ---


,origin_league_id,name,age_at_transfer,einkaufspreis,peak_marktwert,roi_score
159,ARG1,Piero Hincapié,19,6350000.0,50000000,36.179030
37,ARG1,Santiago Ascacíbar,20,6000000.0,25000000,15.833333
67,ARG1,Nico González,20,11260000.0,40000000,12.761989
18,ARG1,Exequiel Palacios,22,17000000.0,45000000,7.486631
197,ARG1,Julián Malatini,23,2000000.0,3000000,2.173913
114,ARG1,Leonardo Balerdi,20,15500000.0,20000000,1.451613
533,ARG1,Marcelo Saracchi,20,12000000.0,14000000,0.833333
337,ARG1,Lucas Alario,25,19000000.0,20000000,0.210526
481,BRA1,Joelinton,19,2200000.0,42000000,95.215311
494,BRA1,William,22,5000000.0,14000000,8.181818


In [ ]:
# Problem: Mittelwert-Orientierung des ROI-Scores zeigt, dass einzelne Ausreißer (z.B. Norwegen/Ryerson, Brasilien/Joelinton) die Platzierung einzelner Ligen extrem beeinflusst.
# Lösung: Ausweichen auf Medien statt Mittelwert, um eher zu zeigen, was ein Bundesliga-Verein im "Normalfall" von einem Transfer aus einer bestimmten Liga erwarten kann.

# Insights: Japan, Serbien und Tschechien mit homogener Preisstruktur bzw. guten ROI-Scores auffällig.

# Next Step: Medien statt Durchschnitt für ROI-Scores

In [6]:
# 1. Sicherstellen, dass nur Ligen mit mindestens 6 Transfers enthalten sind
league_counts = df_clean['origin_league_id'].value_counts()
leagues_to_keep = league_counts[league_counts >= 6].index
df_filtered = df_clean[df_clean['origin_league_id'].isin(leagues_to_keep)].copy()

# 2. Aggregation: Mittelwert UND Median berechnen
league_final = df_filtered.groupby('origin_league_id').agg({
    'player_id': 'count',
    'roi_score': ['mean', 'median'],
    'einkaufspreis': 'median',
    'peak_marktwert': 'median'
})

# Spaltennamen flach machen (Multi-Index auflösen)
league_final.columns = ['Anzahl', 'ROI_Score_Mittelwert', 'ROI_Score_Median', 'Einkauf_Median', 'Peak_MW_Median']

# 3. Nach Median sortieren (da dies unsere "ehrlichere" Metrik ist)
league_final = league_final.sort_values('ROI_Score_Median', ascending=False)

# Formatierung
league_final['ROI_Score_Mittelwert'] = league_final['ROI_Score_Mittelwert'].round(2)
league_final['ROI_Score_Median'] = league_final['ROI_Score_Median'].round(2)
league_final['Einkauf_Median'] = league_final['Einkauf_Median'].map('{:,.0f} €'.format)
league_final['Peak_MW_Median'] = league_final['Peak_MW_Median'].map('{:,.0f} €'.format)

print("LIGA-CHECK: Mittelwert vs. Median (Sortiert nach Median)")
display(league_final)

LIGA-CHECK: Mittelwert vs. Median (Sortiert nach Median)


,Anzahl,ROI_Score_Mittelwert,ROI_Score_Median,Einkauf_Median,Peak_MW_Median
origin_league_id,,,,,
C1,40,9.35,7.96,"3,350,000 €","7,250,000 €"
NL1,48,12.16,7.02,"3,000,000 €","8,000,000 €"
ES1,24,12.92,6.91,"6,750,000 €","18,500,000 €"
BE1,40,17.89,5.95,"3,750,000 €","9,500,000 €"
PO1,15,16.06,5.85,"7,180,000 €","17,000,000 €"
A1,50,12.46,5.32,"3,250,000 €","8,000,000 €"
FR1,84,13.71,4.97,"7,750,000 €","18,000,000 €"
ARG1,8,9.62,4.83,"11,630,000 €","22,500,000 €"
DK1,33,11.09,4.80,"2,500,000 €","4,500,000 €"


In [ ]:
#           Kategorie	                Logik (ROI Median)	    Ligen	                        Kommentar
# Tier 1:   Efficiency Peaks	        Median > 6.0	        C1, NL1, ES1, BE1	            "Sichere Häfen für Wertsteigerung"
# Tier 2:   Solid Development	        Median 4.0 - 6.0	    PO1, A1, FR1, DK1, NO1, PL1	    "Klassische Ausbildungsmärkte"
# Tier 3:   High Capital / Low ROI	    Median 2.0 - 4.0	    GB1, IT1, BRA1, JAP1, SE1	    "Premium-Märkte / Risiko-Invests"
# Tier 4:   Speculative / Low Yield	    Median < 2.0	        TS1, TR1, GR1	                "Vorsicht: Hohe Streuverluste"